# 07 — Choosing the answering model

The application answers with **Qwen2.5-1.5B-Instruct**. Until 2026-09-21 the short
answers and quiz questions came from **google/flan-t5-large**, and **qwen3:14b**
(through Ollama) was measured as the larger alternative. This notebook sets the
three side by side from the runs saved by the other notebooks.

**Re-running.** Nothing here is recomputed. Each comparison is a run of another
notebook with a different model, chosen with environment variables set in that
notebook's first cell before anything is imported:

| model | settings |
|---|---|
| flan-t5-large | `SA_MODEL=google/flan-t5-large` |
| Qwen2.5-1.5B (default) | none |
| qwen3:14b | `SA_BACKEND=ollama`, `SA_MODEL=qwen3:14b` |

QASPER is notebook 01, the 25-question analysis notebook 01, quiz generation and
the grader notebook 09.

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath("") if os.path.basename(os.path.abspath("")) == "notebooks"
                else os.path.join(os.path.abspath(""), "notebooks"))
from eval_common import repo_root  # noqa: E402

REPO_ROOT = repo_root()
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "notebooks"))

# The application's settings (backend/service.py). They are read when the
# pipeline modules are imported, so they are set before anything else.
os.environ.setdefault("SA_EMBEDDER", "bge-small")

# False: show the results saved in data/eval. True: run the evaluation again
# and overwrite them (the run time is given at the top of the notebook).
RUN = False

In [2]:
import json

import pandas as pd

EVAL_DIR = REPO_ROOT / "data" / "eval"


def saved(name):
    """A results file from data/eval."""
    return json.loads((EVAL_DIR / name).read_text(encoding="utf-8"))

## QASPER dev set (281 papers, 1,005 questions)

In [3]:
rows = []
for name in ("qasper_dev.json", "qasper_dev_qwen3-14b.json", "qasper_dev_qwen3-14b_abstain-off.json"):
    r = saved(name)
    rows.append({"model": r["model"], "abstain": r["abstain"],
                 "answer F1": round(r["answer_f1"], 3), "evidence F1": round(r["evidence_f1"], 3),
                 "minutes": round(r["seconds"] / 60, 1)})
pd.DataFrame(rows)

,model,abstain,answer F1,evidence F1,minutes
0,Qwen/Qwen2.5-1.5B-Instruct,check,0.250,0.292,12.0
1,qwen3:14b,check,0.336,0.311,83.8
2,qwen3:14b,off,0.339,0.248,67.9


The paper's fine-tuned baseline scores 29.05 answer F1 on dev (28.01 with the evidence scaffold; notebook 01). Zero-shot, the 14B exceeds it by about 4.5 points and the 1.5B falls about 4 points short, while running about seven times faster.

## Our 25 questions, shipped retrieval (sentence, bge-small, hybrid, k = 3)

In [4]:
rows = []
for name, model in (("generation_analysis_sentence_bge-small_hybrid.json", "flan-t5-large"),
                    ("generation_analysis_sentence_bge-small_hybrid_qwen2.5-1.5b.json", "Qwen2.5-1.5B")):
    r = saved(name)
    correct = sum(x["answer_correct"] for x in r["results"])
    rows.append({"model": model, "correct": f"{correct}/{r['total_questions']}",
                 **{k.split(" (")[0]: v for k, v in r["bucket_counts"].items()}})
pd.DataFrame(rows).set_index("model")

,correct,1. retrieved + correct answer,2. retrieved + wrong answer,3. not retrieved + wrong answer,4. not retrieved + correct answer
model,,,,,
flan-t5-large,20/25,18,3,2,2
Qwen2.5-1.5B,18/25,16,5,2,2


## Quiz questions and grading

In [5]:
rows = []
for name, model in (("quiz_generation_flan.json", "flan-t5-large"), ("quiz_generation.json", "Qwen2.5-1.5B")):
    s = saved(name)["summary"]
    rows.append({"model": model, "attempts": s["attempts"], "kept": s["outcomes"].get("kept"),
                 "kept rate": s["kept_rate"], "well-formed rate": s["well_formed_rate"],
                 "topics with an item": s["topics_with_at_least_one_item"],
                 "seconds per attempt": s["seconds_per_attempt"]})
display(pd.DataFrame(rows).set_index("model"))
rows = []
for name, model in (("grader_calibration.json", "flan-t5-large answers"),
                    ("grader_calibration_sentence_bge-small_hybrid_qwen2.5-1.5b.json", "Qwen2.5-1.5B answers")):
    r = saved(name)
    rows.append({"graded": model, **r["at_threshold"]})
pd.DataFrame(rows).set_index("graded")

,attempts,kept,kept rate,well-formed rate,topics with an item,seconds per attempt
model,,,,,,
flan-t5-large,86,26,0.302,0.884,18,3.27
Qwen2.5-1.5B,86,29,0.337,0.872,22,1.95


,threshold,tp,fp,tn,fn,accuracy,precision,recall,f1
graded,,,,,,,,,
flan-t5-large answers,0.7,11,0,38,1,0.98,1.000,0.917,0.957
Qwen2.5-1.5B answers,0.7,14,1,31,4,0.90,0.933,0.778,0.848
